# Classification Results Figures & Table

This notebook produces all publication-quality visuals for Section 2.3:
- **Table 2**: Main classification results (LaTeX-ready)
- **Figure 3**: Rationale augmentation effect (dumbbell plot)
- **Figure 4**: WithUpdate vs Random (forest plot)
- **Figure S1**: Sensitivity to exemplar count m (supplementary)

Styling matches `fig_selection_efficiency` notebook conventions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 7.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'lines.linewidth': 1.3,
    'lines.markersize': 4,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

In [ ]:
# ── Shared constants ────────────────────────────────────────────

# TODO: Adjust this to your repo root if needed
REPO_ROOT = Path('.').resolve()
root_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(
    (candidate.resolve() for candidate in root_candidates
     if (candidate / 'results').exists() and (candidate / 'notebooks').exists()),
    Path.cwd().resolve(),
)
print(f'Using repo root: {REPO_ROOT}')

OUTPUT_DIR = REPO_ROOT / 'notebooks'
OUTPUT_DIR.mkdir(exist_ok=True)

# TODO: Point to the consolidated tables produced by evaluations_loo_consolidated_phiV
CONSOLIDATED_DIR = REPO_ROOT / 'analysis' / 'consolidated_loo_tables'
assert CONSOLIDATED_DIR.exists(), f'Missing {CONSOLIDATED_DIR}'

# Load all consolidated tables
table_1 = pd.read_csv(CONSOLIDATED_DIR / 'table_1.csv')   # Main results
table_3 = pd.read_csv(CONSOLIDATED_DIR / 'table_3.csv')   # Pairwise tests
table_4 = pd.read_csv(CONSOLIDATED_DIR / 'table_4.csv')   # Sensitivity (raw seeds)
table_5 = pd.read_csv(CONSOLIDATED_DIR / 'table_5.csv')   # Aggregated summary

print(f'table_1: {len(table_1)} rows')
print(f'table_3: {len(table_3)} rows')
print(f'table_4: {len(table_4)} rows')
print(f'table_5: {len(table_5)} rows')

In [ ]:
# ── Shared display settings ─────────────────────────────────────

MODEL_DISPLAY_NAMES = {
    'meta-llama/Llama-3.2-3B-Instruct': 'Llama-3.2-3B',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16': 'Ministral-3B',
    'google/gemma-3-4b-it': 'Gemma-3-4B',
    'Qwen/Qwen3-4B-Instruct-2507': 'Qwen3-4B',
    'microsoft/MediPhi-Instruct': 'MediPhi',
    'microsoft/Phi-3.5-mini-instruct': 'Phi-3.5-mini',
    'microsoft/Phi-4-mini-instruct': 'Phi-4-mini',
    'COMMITTEE': 'Committee',
    'regex_baseline': 'Regex baseline',
}

# Order: weakest zero-shot first → strongest, then committee, then regex
MODEL_ORDER = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'microsoft/Phi-4-mini-instruct',
    'microsoft/MediPhi-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16',
    'google/gemma-3-4b-it',
    'Qwen/Qwen3-4B-Instruct-2507',
    'COMMITTEE',
    'regex_baseline',
]

# Colourblind-safe palette (consistent across all figures)
COLORS = {
    'zero_shot':  '#111111',
    'label_only': '#0072B2',
    'rationale':  '#009E73',
    'random':     '#6c757d',
    'regex':      '#D55E00',
    'WithUpdate': '#D55E00',
    'Random':     '#6c757d',
    'sig_pos':    '#009E73',   # significant positive
    'sig_neg':    '#D55E00',   # significant negative  
    'nonsig':     '#999999',
}

def short_name(model):
    return MODEL_DISPLAY_NAMES.get(model, model)

---
## Table 2: Main Classification Results

Long-prompt variant only (main text). Short-prompt goes to supplementary.

Format: F1 [95% CI]. Bold best per model-row.

In [ ]:
def format_f1_ci(row):
    """Format as '0.XXX [0.XXX, 0.XXX]' or '—' if NaN."""
    if pd.isna(row['macro_f1']):
        return '—'
    return f"{row['macro_f1']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}]"


def build_main_table(df, prompt_variant='long'):
    """
    Build the main results table for a given prompt variant.
    Columns: Model | Zero-shot | WU Label | WU Rationale | Rand Label | Rand Rationale
    Two panels: MIMIC, Indian.
    """
    # Filter to requested prompt variant (or NaN for regex/zero-shot)
    mask = (
        (df['prompt_variant'] == prompt_variant) |
        df['prompt_variant'].isna()
    )
    sub = df[mask].copy()
    
    panels = {}
    for cohort in ['MIMIC', 'Indian']:
        cohort_df = sub[sub['cohort'] == cohort].copy()
        
        # Also include regex which has NaN cohort indicator in prompt_variant
        regex_rows = sub[
            (sub['model'] == 'regex_baseline') & 
            (sub['cohort'] == cohort)
        ]
        
        rows = []
        for model in MODEL_ORDER:
            model_df = cohort_df[cohort_df['model'] == model]
            
            # Zero-shot
            zs = model_df[model_df['prompt_regime'] == 'zero-shot']
            zs_str = format_f1_ci(zs.iloc[0]) if len(zs) > 0 else '—'
            
            # WithUpdate label-only
            wu_lo = model_df[
                (model_df['regime'] == 'WithUpdate') & 
                (model_df['prompt_regime'] == 'label-only ICL')
            ]
            wu_lo_str = format_f1_ci(wu_lo.iloc[0]) if len(wu_lo) > 0 else '—'
            
            # WithUpdate rationale
            wu_ra = model_df[
                (model_df['regime'] == 'WithUpdate') & 
                (model_df['prompt_regime'] == 'rationale-augmented ICL')
            ]
            wu_ra_str = format_f1_ci(wu_ra.iloc[0]) if len(wu_ra) > 0 else '—'
            
            # Random label-only
            rd_lo = model_df[
                (model_df['regime'] == 'Random') & 
                (model_df['prompt_regime'] == 'label-only ICL')
            ]
            rd_lo_str = format_f1_ci(rd_lo.iloc[0]) if len(rd_lo) > 0 else '—'
            
            # Random rationale
            rd_ra = model_df[
                (model_df['regime'] == 'Random') & 
                (model_df['prompt_regime'] == 'rationale-augmented ICL')
            ]
            rd_ra_str = format_f1_ci(rd_ra.iloc[0]) if len(rd_ra) > 0 else '—'
            
            # Regex baseline
            if model == 'regex_baseline':
                regex = sub[
                    (sub['model'] == 'regex_baseline') & 
                    (sub['cohort'] == cohort)
                ]
                regex_str = format_f1_ci(regex.iloc[0]) if len(regex) > 0 else '—'
                zs_str = regex_str
                wu_lo_str = wu_ra_str = rd_lo_str = rd_ra_str = '—'
            
            rows.append({
                'Model': short_name(model),
                'Zero-shot': zs_str,
                'WU Label': wu_lo_str,
                'WU Rationale': wu_ra_str,
                'Rand Label': rd_lo_str,
                'Rand Rationale': rd_ra_str,
            })
        
        panels[cohort] = pd.DataFrame(rows)
    
    return panels


panels = build_main_table(table_1, prompt_variant='long')

print('MIMIC-IV (long prompt, m = 4):')
display(panels['MIMIC'])

print('\nIndian OCR (long prompt, m = 4):')
display(panels['Indian'])

In [ ]:
# ── Export Table 2 as LaTeX ──────────────────────────────────────

def to_latex_table(panels, caption, label):
    """Generate LaTeX for the two-panel main results table."""
    lines = []
    lines.append(r'\begin{table}[!ht]')
    lines.append(r'\centering')
    lines.append(r'\small')
    lines.append(r'\caption{' + caption + '}')
    lines.append(r'\label{' + label + '}')
    lines.append(r'\begin{tabular}{l c cc cc}')
    lines.append(r'\toprule')
    lines.append(r' & & \multicolumn{2}{c}{WithUpdate} & \multicolumn{2}{c}{Random} \\\\')
    lines.append(r'\cmidrule(lr){3-4} \cmidrule(lr){5-6}')
    lines.append(r'Model & Zero-shot & Label-only & Rationale & Label-only & Rationale \\\\')
    
    for cohort_name, df in panels.items():
        lines.append(r'\midrule')
        lines.append(r'\multicolumn{6}{l}{\textit{' + cohort_name + r'}} \\\\')
        lines.append(r'\midrule')
        for _, row in df.iterrows():
            cols = ' & '.join([
                row['Model'],
                row['Zero-shot'],
                row['WU Label'],
                row['WU Rationale'],
                row['Rand Label'],
                row['Rand Rationale'],
            ])
            lines.append(cols + r' \\\\')
    
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)


latex_str = to_latex_table(
    panels,
    caption=(
        'Classification performance (macro F1 with 95\\% bootstrap CI) '
        'at $m = 4$ exemplars per class ($k = 12$ total), long prompt variant. '
        'Short-prompt results are provided in Supplementary Table~S1.'
    ),
    label='tab:main-results',
)

table_path = OUTPUT_DIR / 'table_2_main_results.tex'
table_path.write_text(latex_str)
print(f'Saved: {table_path}')
print()
print(latex_str)

---
## Figure 3: Rationale Augmentation Effect (Dumbbell Plot)

Shows label-only → rationale-augmented ICL improvement per model.

Long-prompt variant, WithUpdate selection.

Two panels: (a) MIMIC-IV, (b) Indian OCR.

In [ ]:
def get_f1(df, cohort, prompt_variant, model, regime, prompt_regime):
    """Extract a single macro_f1 value from table_1."""
    mask = (df['cohort'] == cohort) & (df['model'] == model)
    if pd.notna(regime):
        mask = mask & (df['regime'] == regime)
    else:
        mask = mask & df['regime'].isna()
    mask = mask & (df['prompt_regime'] == prompt_regime)
    if prompt_variant is not None:
        mask = mask & (df['prompt_variant'] == prompt_variant)
    else:
        mask = mask & df['prompt_variant'].isna()
    rows = df[mask]
    if len(rows) == 0:
        return np.nan
    return rows.iloc[0]['macro_f1']


def is_significant(df, cohort, prompt_variant, model, comparison):
    """Check significance from table_3."""
    mask = (
        (df['cohort'] == cohort) &
        (df['prompt_variant'] == prompt_variant) &
        (df['model'] == model) &
        (df['comparison'] == comparison)
    )
    rows = df[mask]
    if len(rows) == 0:
        return False
    return rows.iloc[0]['significant'] == True


# Models to show (exclude regex_baseline — it has no ICL results)
DUMBBELL_MODELS = [m for m in MODEL_ORDER if m not in ('regex_baseline',)]

PROMPT_VARIANT = 'long'

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.2, 3.2),
    sharey=True,
    gridspec_kw={'wspace': 0.08},
)

for ax, panel_label, cohort, title in [
    (axes[0], 'a', 'MIMIC', 'MIMIC-IV ($n = 628$)'),
    (axes[1], 'b', 'Indian', 'Indian OCR ($n = 340$)'),
]:
    y_positions = np.arange(len(DUMBBELL_MODELS))
    
    for i, model in enumerate(DUMBBELL_MODELS):
        lo_f1 = get_f1(table_1, cohort, PROMPT_VARIANT, model,
                        'WithUpdate', 'label-only ICL')
        ra_f1 = get_f1(table_1, cohort, PROMPT_VARIANT, model,
                        'WithUpdate', 'rationale-augmented ICL')
        
        sig = is_significant(table_3, cohort, PROMPT_VARIANT, model,
                             'rationale_WithUpdate vs label_only_WithUpdate')
        
        # Connecting line only for significant differences
        if sig:
            ax.plot(
                [lo_f1, ra_f1], [i, i],
                color=COLORS['sig_pos'], linewidth=2.0,
                solid_capstyle='round', zorder=2,
            )
            # Delta annotation
            delta = ra_f1 - lo_f1
            mid_x = max(lo_f1, ra_f1) + 0.01
            ax.text(
                mid_x, i,
                f'+{delta:.2f}',
                fontsize=4.5, color=COLORS['sig_pos'],
                va='center', ha='left', fontweight='bold',
            )
        
        # Label-only dot
        ax.scatter(
            lo_f1, i, color=COLORS['label_only'],
            marker='o', s=25, zorder=3, edgecolors='white', linewidths=0.3,
        )
        # Rationale dot
        ax.scatter(
            ra_f1, i, color=COLORS['rationale'],
            marker='D', s=25, zorder=3, edgecolors='white', linewidths=0.3,
        )
    
    # Regex baseline reference
    regex_f1 = get_f1(table_1, cohort, None, 'regex_baseline', np.nan, np.nan)
    if not np.isnan(regex_f1):
        ax.axvline(
            regex_f1, color=COLORS['regex'],
            linestyle='--', linewidth=1.0, alpha=0.7, zorder=1,
        )
        ax.text(
            regex_f1 + 0.005, len(DUMBBELL_MODELS) - 0.5,
            'regex', fontsize=6, color=COLORS['regex'], va='bottom',
        )
    
    ax.set_yticks(y_positions)
    ax.set_yticklabels([short_name(m) for m in DUMBBELL_MODELS], fontsize=7.5)
        
    ax.set_xlabel('Macro F1')
    ax.set_title(title, pad=8)
    ax.grid(axis='x', linewidth=0.3, alpha=0.5)
    ax.set_ylim(-0.7, len(DUMBBELL_MODELS) - 0.3)
    ax.invert_yaxis()

    ax.text(
        -0.08, 1.04, panel_label,
        transform=ax.transAxes, fontsize=12, fontweight='bold',
        va='bottom', ha='left',
    )

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['label_only'],
           markersize=5, label='Label-only ICL'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor=COLORS['rationale'],
           markersize=5, label='Rationale-augmented ICL'),
    Line2D([0], [0], color=COLORS['sig_pos'], linewidth=2.0,
           label='Significant improvement'),
]
axes[0].legend(
    handles=legend_elements, loc='lower left',
    frameon=True, fancybox=False, edgecolor='#cccccc',
    framealpha=0.95, borderpad=0.4, fontsize=6.5,
)

fig.savefig(OUTPUT_DIR / 'fig_rationale_effect.pdf')
fig.savefig(OUTPUT_DIR / 'fig_rationale_effect.png')
print(f'Saved: {OUTPUT_DIR / "fig_rationale_effect.pdf"}')
plt.show()

---
## Figure 4: WithUpdate vs Random (Forest Plot)

Δ macro F1 (WithUpdate − Random) under rationale-augmented ICL.

Long-prompt variant. Two panels: (a) MIMIC-IV, (b) Indian OCR.

In [ ]:
# Check the two suspicious rows
mask = (
    (table_3['cohort'] == 'Indian') &
    (table_3['prompt_variant'] == 'long') &
    (table_3['comparison'] == 'rationale_WithUpdate vs rationale_Random')
)
print(table_3[mask][['model', 'delta_f1', 'significant', 'bootstrap_p']].to_string())

In [ ]:
# =============================================================
# Figure: Proposed selection vs Random (Dumbbell Plot)
# =============================================================

FOREST_MODELS = [m for m in MODEL_ORDER if m not in ('regex_baseline',)]
COMPARISON_KEY = 'rationale_WithUpdate vs rationale_Random'
PROMPT_VARIANT = 'long'

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.2, 3.6),
    sharey=True,
    gridspec_kw={'wspace': 0.12},
)

for ax, panel_label, cohort, title in [
    (axes[0], 'a', 'MIMIC', 'MIMIC-IV ($n = 628$)'),
    (axes[1], 'b', 'Indian', 'Indian OCR ($n = 340$)'),
]:
    y_positions = np.arange(len(FOREST_MODELS))

    all_points = []

    for i, model in enumerate(FOREST_MODELS):

        # Zero-shot F1 (grey diamond, absolute)
        zs_f1 = get_f1(table_1, cohort, PROMPT_VARIANT, model, np.nan, 'zero-shot')
        if not np.isnan(zs_f1):
            ax.scatter(
                zs_f1, i, color='#bbbbbb', marker='d', s=18,
                zorder=2, linewidths=0.3, edgecolors='#999999',
            )
            all_points.append(zs_f1)

        # Proposed selection rationale F1 (coloured circle, absolute)
        wu_f1 = get_f1(table_1, cohort, PROMPT_VARIANT, model,
                        'WithUpdate', 'rationale-augmented ICL')
        if not np.isnan(wu_f1):
            ax.scatter(
                wu_f1, i, color=COLORS['rationale'], marker='o', s=28,
                zorder=4, edgecolors='white', linewidths=0.4,
            )
            all_points.append(wu_f1)

        # Random rationale F1 (grey square, absolute)
        rd_f1 = get_f1(table_1, cohort, PROMPT_VARIANT, model,
                        'Random', 'rationale-augmented ICL')
        if not np.isnan(rd_f1):
            ax.scatter(
                rd_f1, i, color=COLORS['random'], marker='s', s=22,
                zorder=3, edgecolors='white', linewidths=0.3,
            )
            all_points.append(rd_f1)

        # Connecting line + delta annotation only for significant differences
        if not np.isnan(wu_f1) and not np.isnan(rd_f1):
            row = table_3[
                (table_3['cohort'] == cohort) &
                (table_3['prompt_variant'] == PROMPT_VARIANT) &
                (table_3['model'] == model) &
                (table_3['comparison'] == COMPARISON_KEY)
            ]
            if len(row) > 0 and not pd.isna(row.iloc[0]['delta_f1']):
                sig = row.iloc[0]['significant']
                delta = row.iloc[0]['delta_f1']

                if sig:
                    line_color = COLORS['sig_pos'] if delta > 0 else COLORS['sig_neg']
                    ax.plot(
                        [wu_f1, rd_f1], [i, i],
                        color=line_color, linewidth=2.0,
                        solid_capstyle='round', zorder=2,
                    )
                    # Delta annotation
                    sign = '+' if delta > 0 else ''
                    mid_x = max(wu_f1, rd_f1) + 0.01
                    ax.text(
                        mid_x, i,
                        f'{sign}{delta:.2f}',
                        fontsize=5.0, color=line_color,
                        va='center', ha='left', fontweight='bold',
                        clip_on=False,
                    )

    # Regex baseline vertical line
    regex_rows = table_1[
        (table_1['cohort'] == cohort) &
        (table_1['model'] == 'regex_baseline')
    ]
    if len(regex_rows) > 0:
        regex_f1 = regex_rows.iloc[0]['macro_f1']
        if not np.isnan(regex_f1):
            ax.axvline(
                regex_f1, color=COLORS['regex'],
                linestyle='--', linewidth=1.0, alpha=0.6, zorder=1,
            )
            ax.text(
                regex_f1, -0.6, 'regex',
                fontsize=5.0, color=COLORS['regex'],
                ha='center', va='top',
            )
            all_points.append(regex_f1)

    # Alternating background bands
    for i in range(len(FOREST_MODELS)):
        if i % 2 == 0:
            ax.axhspan(i - 0.4, i + 0.4, color='#f7f9fb', zorder=0)

    # Axis formatting
    ax.set_yticks(y_positions)
    ax.set_yticklabels([short_name(m) for m in FOREST_MODELS], fontsize=7.5)

    ax.set_xlabel('Macro F1')
    ax.set_title(title, pad=8)
    ax.grid(axis='x', linewidth=0.3, alpha=0.5)
    ax.set_ylim(-0.8, len(FOREST_MODELS) - 0.2)
    ax.invert_yaxis()

    # Auto-scale x with padding for delta annotations
    if all_points:
        xmin = max(0, min(all_points) - 0.05)
        xmax = min(1.08, max(all_points) + 0.09)
        ax.set_xlim(xmin, xmax)

    # Panel label
    ax.text(
        -0.08, 1.04, panel_label,
        transform=ax.transAxes, fontsize=12, fontweight='bold',
        va='bottom', ha='left',
    )

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='d', color='w', markerfacecolor='#bbbbbb',
           markeredgecolor='#999999', markersize=5, label='Zero-shot'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['rationale'],
           markersize=5.5, label='Proposed selection + rationale'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=COLORS['random'],
           markersize=5, label='Random + rationale'),
    Line2D([0], [0], color=COLORS['sig_pos'], linewidth=2.0,
           label='Sig. difference'),
    Line2D([0], [0], color=COLORS['regex'], linestyle='--', linewidth=1.0,
           label='Regex baseline'),
]
axes[1].legend(
    handles=legend_elements, loc='lower left',
    frameon=True, fancybox=False, edgecolor='#cccccc',
    framealpha=0.95, borderpad=0.4, fontsize=6,
    ncol=1,
)

fig.savefig(OUTPUT_DIR / 'fig_wu_vs_random.pdf')
fig.savefig(OUTPUT_DIR / 'fig_wu_vs_random.png')
print(f'Saved: {OUTPUT_DIR / "fig_wu_vs_random.pdf"}')
plt.show()

---
## Figure S1: Sensitivity to Exemplar Count (Supplementary)

Per-model line plots of macro F1 vs m ∈ {2, 4, 6, 8, 10}.

Three lines per panel: WithUpdate label-only, WithUpdate rationale, Random rationale.

Shaded band = ±1 SD across seeds.

Horizontal references: zero-shot F1 (dashed), regex baseline (dotted).

In [ ]:
SENSITIVITY_MODELS = [m for m in MODEL_ORDER if m not in ('COMMITTEE', 'regex_baseline')]

STRATEGY_STYLES = {
    ('WithUpdate', 'label-only ICL'): {
        'color': COLORS['label_only'], 'marker': 'o',
        'ls': '-', 'label': 'WU label-only',
    },
    ('WithUpdate', 'rationale-augmented ICL'): {
        'color': COLORS['rationale'], 'marker': 'D',
        'ls': '-', 'label': 'WU rationale',
    },
    ('Random', 'rationale-augmented ICL'): {
        'color': COLORS['random'], 'marker': 's',
        'ls': '--', 'label': 'Random rationale',
    },
}


def plot_sensitivity(cohort, prompt_variant, suptitle):
    """One figure per cohort × prompt variant."""
    n_models = len(SENSITIVITY_MODELS)
    n_cols = 4
    n_rows = int(np.ceil(n_models / n_cols))
    
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(7.2, 2.2 * n_rows),
        sharex=True, sharey=False,
    )
    axes_flat = axes.ravel()
    
    sub = table_4[
        (table_4['cohort'] == cohort) &
        (table_4['prompt_variant'] == prompt_variant)
    ].copy()
    
    for idx, model in enumerate(SENSITIVITY_MODELS):
        ax = axes_flat[idx]
        model_df = sub[sub['model'] == model]
        
        for (regime, prompt_regime), style in STRATEGY_STYLES.items():
            strat_df = model_df[
                (model_df['regime'] == regime) &
                (model_df['prompt_regime'] == prompt_regime)
            ]
            if strat_df.empty:
                continue
            
            agg = strat_df.groupby('m')['macro_f1'].agg(['mean', 'std']).reset_index()
            agg['std'] = agg['std'].fillna(0)
            
            ax.plot(
                agg['m'], agg['mean'],
                color=style['color'], marker=style['marker'],
                linestyle=style['ls'], linewidth=1.3,
                markerfacecolor='white', markeredgewidth=0.8,
                label=style['label'] if idx == 0 else None,
            )
            ax.fill_between(
                agg['m'],
                np.clip(agg['mean'] - agg['std'], 0, 1),
                np.clip(agg['mean'] + agg['std'], 0, 1),
                color=style['color'], alpha=0.10,
            )
        
        # Zero-shot reference
        zs_f1 = get_f1(table_1, cohort, prompt_variant, model, np.nan, 'zero-shot')
        if not np.isnan(zs_f1):
            ax.axhline(zs_f1, color='#111111', linestyle='--', linewidth=0.7, alpha=0.5)
        
        # Regex reference
        regex_f1 = get_f1(table_1, cohort, None, 'regex_baseline', np.nan, np.nan)
        if not np.isnan(regex_f1):
            ax.axhline(regex_f1, color=COLORS['regex'], linestyle=':', linewidth=0.7, alpha=0.5)
        
        ax.set_title(short_name(model), fontsize=8, pad=4)
        ax.set_xticks([2, 4, 6, 8, 10])
        ax.grid(axis='y', linewidth=0.3, alpha=0.4)
        ax.tick_params(labelsize=6.5)
    
    # Hide unused panels
    for idx in range(n_models, len(axes_flat)):
        axes_flat[idx].axis('off')
    
    # Shared labels
    fig.supxlabel('Exemplars per class ($m$)', fontsize=9)
    fig.supylabel('Macro F1', fontsize=9)
    
    # Legend from first panel
    handles, labels = axes_flat[0].get_legend_handles_labels()
    if handles:
        fig.legend(
            handles, labels,
            loc='upper center', ncol=3, frameon=False,
            fontsize=7, bbox_to_anchor=(0.5, 1.02),
        )
    
    fig.suptitle(suptitle, fontsize=10, y=1.06)
    fig.tight_layout()
    return fig


# MIMIC long
fig_mimic = plot_sensitivity('MIMIC', 'long', 'MIMIC-IV — Sensitivity to $m$ (long prompt)')
fig_mimic.savefig(OUTPUT_DIR / 'fig_sensitivity_mimic_long.pdf')
fig_mimic.savefig(OUTPUT_DIR / 'fig_sensitivity_mimic_long.png')
plt.show()

# Indian long
fig_indian = plot_sensitivity('Indian', 'long', 'Indian OCR — Sensitivity to $m$ (long prompt)')
fig_indian.savefig(OUTPUT_DIR / 'fig_sensitivity_indian_long.pdf')
fig_indian.savefig(OUTPUT_DIR / 'fig_sensitivity_indian_long.png')
plt.show()

print('Sensitivity figures saved.')

In [ ]:
# ── Summary of all outputs ──────────────────────────────────────

print('=== Generated outputs ===')
for f in sorted(OUTPUT_DIR.glob('fig_*')):
    print(f'  {f.name}')
for f in sorted(OUTPUT_DIR.glob('table_*')):
    print(f'  {f.name}')